In [15]:
import numpy as np
import torch

from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

In [2]:
ds_toxigen = load_dataset(
    "toxigen/toxigen-data",
    split="train"
)

def preprocess_toxigen(example):
    label = 1 if example["toxicity_human"] >= 3 else 0
    return {"text": example["text"], "label": label}

ds_toxigen = ds_toxigen.map(
    preprocess_toxigen,
    remove_columns=ds_toxigen.column_names
)

print(ds_toxigen)


Dataset({
    features: ['text', 'label'],
    num_rows: 8960
})


In [3]:
ds_multi_en = load_dataset(
    "textdetox/multilingual_toxicity_dataset",
    split="en"
)

ds_multi_ru = load_dataset(
    "textdetox/multilingual_toxicity_dataset",
    split="ru"
)

def preprocess_multi(example):
    return {"text": example["text"], "label": example["toxic"]}

ds_multi_en = ds_multi_en.map(
    preprocess_multi,
    remove_columns=ds_multi_en.column_names
)

ds_multi_ru = ds_multi_ru.map(
    preprocess_multi,
    remove_columns=ds_multi_ru.column_names
)

ds_multi = concatenate_datasets([ds_multi_en, ds_multi_ru])

print(ds_multi)


Dataset({
    features: ['text', 'label'],
    num_rows: 10000
})


In [4]:
ds_ru1 = load_dataset(
    "Xeonil/ru-merged-toxic-comments",
    split="train"
)

def preprocess_ru1(example):
    return {"text": example["text"], "label": example["target"]}

ds_ru1 = ds_ru1.map(
    preprocess_ru1,
    remove_columns=ds_ru1.column_names
)

print(ds_ru1)


Dataset({
    features: ['text', 'label'],
    num_rows: 271410
})


In [5]:
ds_ru2 = load_dataset(
    "AlexSham/Toxic_Russian_Comments",
    split="train"
)

def preprocess_ru2(example):
    return {"text": example["text"], "label": example["label"]}

ds_ru2 = ds_ru2.map(
    preprocess_ru2,
    remove_columns=ds_ru2.column_names
)

print(ds_ru2)


Dataset({
    features: ['text', 'label'],
    num_rows: 223461
})


In [6]:
full_dataset = concatenate_datasets([
    ds_toxigen,
    ds_multi,
    ds_ru1,
    ds_ru2
])

full_dataset = full_dataset.shuffle(seed=42)
full_dataset = full_dataset.train_test_split(test_size=0.1)

print(full_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 462447
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 51384
    })
})


In [7]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Users\itsuki\anaconda3\envs\transformers5\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\itsuki\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = full_dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")

Map:   0%|          | 0/462447 [00:00<?, ? examples/s]

Map:   0%|          | 0/51384 [00:00<?, ? examples/s]

In [9]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=labels
        )["f1"]
    }

In [13]:
training_args = TrainingArguments(
    output_dir="./toxic_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none"
)

In [16]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.085201,0.116843,0.969504,0.924595
2,0.075212,0.073417,0.981006,0.951127
3,0.078340,0.072878,0.984256,0.959422


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=86709, training_loss=0.08876298696404517, metrics={'train_runtime': 8621.6942, 'train_samples_per_second': 160.913, 'train_steps_per_second': 10.057, 'total_flos': 9.125618866354944e+16, 'train_loss': 0.08876298696404517, 'epoch': 3.0})

In [20]:
def predict_toxicity(text: str):
    model.eval()

    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    return {
        "non_toxic": probs[0][0].item(),
        "toxic": probs[0][1].item()
    }

In [23]:
trainer.save_model("./toxic_model")
tokenizer.save_pretrained("./toxic_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./toxic_model\\tokenizer_config.json', './toxic_model\\tokenizer.json')